# 3.9 — Research Agent (Capstone)

This is the Week 3 capstone — a full **Research Agent** that:

1. Takes a research topic from the user
2. Searches for facts and data (multiple tool calls)
3. Drafts a structured report
4. Saves the report to a file

```
User: "Write a report on Artificial Intelligence"
    ↓
Agent: search_topic('AI history')
    ↓
Agent: search_topic('AI applications')
    ↓
Agent: get_statistics('AI market size')
    ↓
Agent: write_report(title, sections)
    ↓
Agent: save_report(filename, content)
    ↓
"Report saved to ai_report.txt"
```

This notebook combines everything from Week 3:
- Multi-tool agent loop (3.2)
- Conversation memory (3.3)
- Multi-step planning (3.1, 3.4)
- LangGraph structure (3.5)
- Multi-agent pattern (3.6)

In [ ]:
!pip install langchain langchain-ollama langgraph --quiet

## Step 1 — Define the Research Toolkit

In [ ]:
from langchain_core.tools import tool
from datetime import date
import os

# Knowledge base — simulates web search results
KNOWLEDGE_BASE = {
    'artificial intelligence': {
        'history':       'AI was coined by John McCarthy in 1956 at the Dartmouth Conference. Key milestones: expert systems (1980s), deep learning breakthrough (2012), transformer models (2017), ChatGPT (2022).',
        'applications':  'AI is used in healthcare (diagnosis), finance (fraud detection), transportation (self-driving cars), education (personalised learning), and entertainment (recommendation systems).',
        'challenges':    'Key challenges include bias in training data, lack of interpretability (black box problem), high energy consumption, job displacement concerns, and AI safety/alignment.',
        'future':        'Future trends include AGI research, multimodal AI, AI agents, smaller efficient models (edge AI), and tighter AI regulation worldwide.',
        'statistics':    'Global AI market: $200B (2023), projected $1.8T by 2030. 77% of devices use AI features. 97M new jobs expected by 2025 due to AI.',
    },
    'python': {
        'history':       'Python was created by Guido van Rossum and released in 1991. Named after Monty Python. Python 3 released in 2008.',
        'applications':  'Python is used in web development (Django, Flask), data science (pandas, numpy), AI/ML (TensorFlow, PyTorch), automation, and scientific computing.',
        'challenges':    'Python is slower than compiled languages. GIL limits multi-threading. High memory usage compared to C/C++.',
        'future':        'Python continues to grow as the dominant language for AI. Python 3.12+ brings significant performance improvements.',
        'statistics':    'Python used by 48% of developers. #1 language on GitHub. 8M+ packages on PyPI. 10M+ active users worldwide.',
    },
    'langchain': {
        'history':       'LangChain was created by Harrison Chase in October 2022. Rapidly became the most popular LLM framework with 80k+ GitHub stars.',
        'applications':  'LangChain is used to build chatbots, RAG systems, AI agents, document analysis tools, and automated workflows.',
        'challenges':    'LangChain has a steep learning curve. Rapid API changes. Can introduce unnecessary complexity for simple use cases.',
        'future':        'LangGraph (stateful agents), LangSmith (observability), and LangServe (deployment) extend the ecosystem significantly.',
        'statistics':    '80k+ GitHub stars. Used in 100k+ projects. Supports 50+ LLM providers. 1M+ monthly PyPI downloads.',
    },
    'rag': {
        'history':       'RAG was introduced by Lewis et al. (Facebook AI) in 2020. It combines retrieval-based and generation-based NLP approaches.',
        'applications':  'RAG is used in enterprise search, customer support bots, legal document Q&A, medical knowledge bases, and code assistants.',
        'challenges':    'RAG challenges: chunking strategy, retrieval quality, context window limits, latency, and keeping the knowledge base up to date.',
        'future':        'Advanced RAG techniques: multi-hop retrieval, knowledge graphs, agentic RAG, and real-time web-augmented generation.',
        'statistics':    '60% of enterprise LLM deployments use some form of RAG. Reduces hallucination by up to 40% compared to plain LLMs.',
    },
}

@tool
def search_topic(topic: str, aspect: str = 'general') -> str:
    """Search for information about a topic.
    topic: the subject to research (e.g., 'artificial intelligence', 'python', 'langchain')
    aspect: what aspect to look up — 'history', 'applications', 'challenges', 'future', or 'statistics'
    """
    topic_lower = topic.lower().strip()
    aspect_lower = aspect.lower().strip()

    # Find matching topic
    for key, data in KNOWLEDGE_BASE.items():
        if key in topic_lower or topic_lower in key:
            if aspect_lower in data:
                return data[aspect_lower]
            # Return all aspects if no specific one
            return '\n'.join(f'{k.title()}: {v}' for k, v in data.items())

    return f'No information found for topic: {topic}'

@tool
def get_statistics(topic: str) -> str:
    """Retrieves statistics and market data for a given topic."""
    for key, data in KNOWLEDGE_BASE.items():
        if key in topic.lower() or topic.lower() in key:
            return data.get('statistics', f'No statistics found for {topic}.')
    return f'No statistics found for: {topic}'

@tool
def save_report(filename: str, content: str) -> str:
    """Saves a research report to a text file. Returns confirmation with the file path."""
    if not filename.endswith('.txt') and not filename.endswith('.md'):
        filename = filename.replace(' ', '_') + '.txt'
    filepath = os.path.join('reports', filename)
    os.makedirs('reports', exist_ok=True)
    with open(filepath, 'w') as f:
        f.write(content)
    return f'Report saved successfully to: {filepath} ({len(content)} characters)'

@tool
def get_today_date() -> str:
    """Returns today's date for including in the report."""
    return date.today().strftime('%B %d, %Y')

# Verify
print('Research toolkit:')
for t in [search_topic, get_statistics, save_report, get_today_date]:
    print(f'  {t.name:<20} — {t.description[:60]}')

## Step 2 — Build the Research Agent

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

llm = ChatOllama(model='llama3.1', temperature=0.3)

tools = [search_topic, get_statistics, save_report, get_today_date]
tool_map = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = """You are an expert research agent. When given a topic:

1. Use search_topic to gather information on multiple aspects:
   - Call it with aspect='history'
   - Call it with aspect='applications'
   - Call it with aspect='challenges'
   - Call it with aspect='future'
2. Use get_statistics to get data and numbers
3. Use get_today_date for the report date
4. Compile all findings into a well-structured report with:
   - Title
   - Executive Summary
   - Sections for each aspect
   - Key Statistics
   - Conclusion
5. Use save_report to save it to disk

Be thorough — search multiple aspects before writing the report."""

def research_agent(topic: str) -> str:
    print(f'Research topic: "{topic}"')
    print('=' * 60)

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f'Research and write a comprehensive report on: {topic}')
    ]

    step = 0
    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            print(f'\nAgent: {response.content}')
            return response.content

        for tc in response.tool_calls:
            step += 1
            tool_name = tc['name']
            tool_args = tc['args']

            result = tool_map[tool_name].invoke(tool_args)

            # Print progress
            if tool_name == 'search_topic':
                print(f'  [{step}] Searching: {tool_args.get("topic", "")} [{tool_args.get("aspect", "")}]')
            elif tool_name == 'save_report':
                print(f'  [{step}] Saving report: {tool_args.get("filename", "")}')
            else:
                print(f'  [{step}] {tool_name}({tool_args})')

            messages.append(ToolMessage(content=str(result), tool_call_id=tc['id']))

print('Research agent ready.')

## Step 3 — Run the Research Agent

In [ ]:
research_agent('Artificial Intelligence')

In [ ]:
# Read and display the saved report
import os

report_files = os.listdir('reports') if os.path.exists('reports') else []
print(f'Reports saved: {report_files}')

if report_files:
    with open(f'reports/{report_files[0]}') as f:
        print(f.read())

In [ ]:
# Try another topic
research_agent('LangChain')

## Step 4 — Interactive Research Session

A loop where the user can request multiple reports.

In [ ]:
def interactive_research_session():
    print('Research Agent — Interactive Mode')
    print('Type a topic to research, or "quit" to exit.\n')

    while True:
        topic = input('Research topic: ').strip()
        if topic.lower() in ('quit', 'exit', 'q'):
            print('Goodbye!')
            break
        if not topic:
            continue
        research_agent(topic)
        print()

# Uncomment to run interactively:
# interactive_research_session()

## Week 3 Summary

| Notebook | Key Concept | What you built |
|----------|------------|----------------|
| 3.1 | Chain vs Agent, ReAct | Basic agent with 3 tools |
| 3.2 | Multi-tool agent | Agent with 5 tools, tool selection |
| 3.3 | Agent memory | Short-term buffer, long-term vector store |
| 3.4 | ReAct from scratch | Manual Thought/Action/Observation loop |
| 3.5 | LangGraph | State graphs, conditional edges, streaming |
| 3.6 | Multi-agent systems | Supervisor + Researcher + Writer agents |
| 3.7 | RAG Agent | Agent that decides when to retrieve |
| 3.8 | Human-in-the-loop | Approval gates for risky actions |
| 3.9 | Research Agent | Full capstone combining all concepts |

**The progression:**
```
Week 1: LLM + Tools
Week 2: LLM + Documents (RAG)
Week 3: LLM + Tools + Memory + Planning + Coordination = Agents
```